In [49]:
%run cochain_complex.ipynb

In [50]:
def compute_ad_kernel(A,d,w=None):
    """Given a basis element A of a Lie algebra g, outputs 
    ker(ad(A)|_{C(d,w)})as a list of cochains, where C is the 
    cochain complex C(g_-,g), g=A.parent
    INPUTS:
    * 'A' - a nonnegative basis element from a Lie algebra g
    * 'd' - a degree
    * 'w' - a weight"""
    if w!=None:
        null_vecs=A.ad_mat(mod='CE',deg=d,wght=w).nullspace()
        return [C.elt({d:{w:v}}) for v in null_vecs]
    
    result={}
    for w in C.basis(d):
        temp=compute_ad_kernel(A,d,w)
        if len(temp)>0: result[w]=temp
    return result

def compute_eigenval(A,c,mod='CE'):
    """Computes the eigenval L so that A(c)=L*c
    INPUTS:
    * 'A' - an element of the Cartan subalgebra of a Lie algebra g
    * 'c' - an element of the Chevalley-Eilenberg complex C(g_-,g)"""

    if c==c.parent.elt({}):
        return 'Null'
    
    d=list(c.vd.keys())[0]
    w=list(c.vd[d].keys())[0]


    Ac=A.ad(c,mod)
    i=0
    temp=c.vd[d][w][i]
    while temp==0:
        i+=1
        temp=c.vd[d][w][i]
    L=Rational(Ac.vd[d][w][i],c.vd[d][w][i])
    temp=Ac-L*c
    if Ac-L*c==c.parent.elt({}): return L
    return None

In [51]:
# This the weighting from the (n-5)th prolongation.
def alt_wght_base(c):
    r=c.wght
    for i in range(len(c.components)-1):
        if str(c.components[i])[0]=='e': r=r-1
        if str(c.components[i])[0]=='N': r=r-2
    if str(c.components[-1])[0]=='e': r=r+1
    if str(c.components[-1])[0]=='N': r=r+2
    return r

def alt_wght(c):
    r=None
    for d in c.vd:
        for w in c.vd[d]:
            for i in range(len(c.parent.basis(d,w))):
                b=c.parent.basis(d,w)[i]
                if c.vd[d][w][i]!=0:
                    if r==None: r=alt_wght_base(b)
                    elif alt_wght_base(b)!=r: return None
    return r

### Temporary

In [57]:
g=Symp_symb(7)
C=cochain_complex(g)
gE=ext_alg(g)
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [ ]:
omega=g.ext_alg.elt({})
for i in range(1,4):
    key = (f'e_{i}',f'e_{7-i}')
    val = (-1)**i
    omega=omega+g.ext_alg.elt_from_cd({key: val})



-(e_1,e_6)+(e_2,e_5)-(e_3,e_4)

In [185]:
d=1
def closed_basis(d,min_w=1):
    elt_list=[]
    for w in C.basis(d):
        if w>=min_w:
            for i in range(len(C.basis(d,w))):
                temp=C.basis(d,w)[i]
                if (temp.components[d].wght<0 and
                        set(map(str,temp.components[0:d])).issubset(set(map(str,[X,e1,e2,e3,e4,e5,e6])))):
                    elt_list.append(temp)

    c = C.elt({})
    for i in range(len(elt_list)):
        key = tuple([str(a) for a in elt_list[i].components])
        val = symbols(f'a{i}')
        c = c + C.elt_from_cd({key:val})

    ccb=c.cb()

    eqs=[]
    if d+1 in ccb.vd:
        for w in ccb.vd[d+1]:
            for i in range(len(ccb.vd[d+1][w])):
                temp=C.basis(d+1,w)[i]
                if (temp.components[d].wght<0 and
                        set(map(str,temp.components[0:d])).issubset(set(map(str,[X,e1,e2,e3,e4,e5,e6])))):
                    if ccb.vd[d+1][w][i]!=0: eqs.append(ccb.vd[d+1][w][i])

    S=solve(eqs)
    c_closed=c.subs(S)
    all_vars=[symbols(f'a{i}') for i in range(len(elt_list))]
    free_vars=[]

    for v in all_vars: 
        if v.xreplace(S)==v: free_vars.append(v)

    r=[]
    for i in range(len(free_vars)):
        temp_subs={a:0 for a in free_vars}
        temp_subs[free_vars[i]]=1
        r.append(c_closed.xreplace(temp_subs))
    return r


In [ ]:
def exact_basis(d,min_w=1):
    

In [198]:
temp_c=C.elt_from_cd({('e_3','X'):1})
temp_c.cb()

-(X,e_2,X)-(e_1,e_3,e_2)-(e_2,e_3,e_3)+(e_3,e_4,e_5)+(e_3,e_5,e_6)

In [202]:
for a in closed_basis(1,-6):
    a.clear_zeros()
    print(a.vd[1].keys(),'-->',a)

dict_keys([0]) --> 2/5*(X,X)-(e_1,e_1)-3/5*(e_2,e_2)-1/5*(e_3,e_3)+1/5*(e_4,e_4)+3/5*(e_5,e_5)+(e_6,e_6)
dict_keys([-1]) --> (e_1,e_2)+(e_2,e_3)+(e_3,e_4)+(e_4,e_5)+(e_5,e_6)
dict_keys([-1]) --> (X,e_2)+(e_6,N)
dict_keys([-2]) --> -(X,e_3)+(e_5,N)
dict_keys([-3]) --> (e_1,e_4)+(e_2,e_5)+(e_3,e_6)
dict_keys([-3]) --> (X,e_4)+(e_4,N)
dict_keys([-4]) --> -(X,e_5)+(e_3,N)
dict_keys([-5]) --> (e_1,e_6)
dict_keys([-5]) --> (X,e_6)+(e_2,N)
dict_keys([-6]) --> (X,N)
dict_keys([-6]) --> (e_1,N)


In [192]:
for c in closed_basis(1,-6):
    print(omega.wedge(c))

-2/5*(X,e_1,e_6,X)+2/5*(X,e_2,e_5,X)-2/5*(X,e_3,e_4,X)-(e_1,e_2,e_5,e_1)-3/5*(e_1,e_2,e_6,e_2)+(e_1,e_3,e_4,e_1)-1/5*(e_1,e_3,e_6,e_3)+1/5*(e_1,e_4,e_6,e_4)+3/5*(e_1,e_5,e_6,e_5)+3/5*(e_2,e_3,e_4,e_2)+1/5*(e_2,e_3,e_5,e_3)-1/5*(e_2,e_4,e_5,e_4)+(e_2,e_5,e_6,e_6)-3/5*(e_3,e_4,e_5,e_5)-(e_3,e_4,e_6,e_6)
(e_1,e_2,e_5,e_2)+(e_1,e_2,e_6,e_3)-(e_1,e_3,e_4,e_2)+(e_1,e_3,e_6,e_4)+(e_1,e_4,e_6,e_5)+(e_1,e_5,e_6,e_6)-(e_2,e_3,e_4,e_3)-(e_2,e_3,e_5,e_4)-(e_2,e_4,e_5,e_5)-(e_3,e_4,e_5,e_6)
-(X,e_1,e_6,e_2)+(X,e_2,e_5,e_2)-(X,e_3,e_4,e_2)+(e_2,e_5,e_6,N)-(e_3,e_4,e_6,N)
(X,e_1,e_6,e_3)-(X,e_2,e_5,e_3)+(X,e_3,e_4,e_3)+(e_1,e_5,e_6,N)-(e_3,e_4,e_5,N)
(e_1,e_2,e_5,e_4)+(e_1,e_2,e_6,e_5)-(e_1,e_3,e_4,e_4)+(e_1,e_3,e_6,e_6)-(e_2,e_3,e_4,e_5)-(e_2,e_3,e_5,e_6)
-(X,e_1,e_6,e_4)+(X,e_2,e_5,e_4)-(X,e_3,e_4,e_4)+(e_1,e_4,e_6,N)-(e_2,e_4,e_5,N)
(X,e_1,e_6,e_5)-(X,e_2,e_5,e_5)+(X,e_3,e_4,e_5)+(e_1,e_3,e_6,N)-(e_2,e_3,e_5,N)
(e_1,e_2,e_5,e_6)-(e_1,e_3,e_4,e_6)
-(X,e_1,e_6,e_6)+(X,e_2,e_5,e_6)-(X,e_3,e_4,e_6)+(e

In [152]:
c_closed

2*a19/5*(X,X)-a19*(e_1,e_1)-3*a19/5*(e_2,e_2)-a19/5*(e_3,e_3)+a19/5*(e_4,e_4)+3*a19/5*(e_5,e_5)+a19*(e_6,e_6)+a26*(X,e_2)+a25*(e_1,e_2)+a25*(e_2,e_3)+a25*(e_3,e_4)+a25*(e_4,e_5)+a25*(e_5,e_6)+a26*(e_6,N)+a45*(X,N)+a46*(e_1,N)+a44*(X,e_6)+a43*(e_1,e_6)+a44*(e_2,N)-a41*(X,e_5)+a41*(e_3,N)+a37*(X,e_4)+a36*(e_1,e_4)+a36*(e_2,e_5)+a36*(e_3,e_6)+a37*(e_4,N)-a32*(X,e_3)+a32*(e_5,N)

In [ ]:
c = C.elt({})

for i in range(1, 7):
    key = ('X',)
    val = symbols('a')
    c = c + C.elt_from_cd({key: val})

    key = (f'e_{i}',)
    val = symbols(f'b_{i}')
    c = c + C.elt_from_cd({key: val})

    key = ('N',)
    val = symbols('f')
    c = c + C.elt_from_cd({key: val})

eqs=[]
ccb=c.cb()

d=1
for w in ccb.vd[d]:
    for i in range(len(ccb.vd[d][w])):
        if ccb.vd[d][w][i]!=0:
            eqs.append(ccb.vd[d][w][i])

solve(eqs)


{a: 0, b_1: 0, b_2: 0, b_3: 0, b_4: 0, b_5: 0, b_6: 0}

In [ ]:
c = C.elt({})

c=c+C.elt_from_cd({('X','X'):symbols('a')})

for i in range(1, 7):
    key = ('X', f'e_{i}')
    val = symbols(f'b_{i}')
    c = c + C.elt_from_cd({key: val})

key = ('X', 'N')
val = symbols('d')
c = c + C.elt_from_cd({key: val})

for i in range(1,7):
    key = (f'e_{i}','N')
    val = symbols(f'f_{i}')
    c = c + C.elt_from_cd({key: val})

    key = (f'e_{i}','X')
    val = symbols(f'g_{i}')
    c = c + C.elt_from_cd({key: val})

    for j in range(1,7):
        key = (f'e_{i}',f'e_{j}')
        val = symbols(f'h_{i}{j}')
        c = c + C.elt_from_cd({key: val})

c=c+C.elt_from_cd({('X','X'):symbols('d')})

eqs=[]
ccb=c.cb()
d=2
for w in ccb.vd[d]:
    for i in range(len(ccb.vd[d][w])):
        if ccb.vd[d][w][i]!=0:
            eqs.append(ccb.vd[d][w][i])

solve(eqs)


{a_1: 0,
 a_2: g_6,
 a_3: -g_5,
 a_4: g_4,
 a_5: -g_3,
 a_6: g_2,
 b_11: -5*d/2,
 b_12: b_56,
 b_13: 0,
 b_14: b_36,
 b_15: 0,
 b_21: 0,
 b_22: -3*d/2,
 b_23: b_56,
 b_24: 0,
 b_25: b_36,
 b_26: 0,
 b_31: 0,
 b_32: 0,
 b_33: -d/2,
 b_34: b_56,
 b_35: 0,
 b_41: 0,
 b_42: 0,
 b_43: 0,
 b_44: d/2,
 b_45: b_56,
 b_46: 0,
 b_51: 0,
 b_52: 0,
 b_53: 0,
 b_54: 0,
 b_55: 3*d/2,
 b_61: 0,
 b_62: 0,
 b_63: 0,
 b_64: 0,
 b_65: 0,
 b_66: 5*d/2,
 f_1: 0,
 f_2: 0,
 f_3: 0,
 f_4: 0,
 f_5: 0,
 f_6: 0}

In [ ]:
c = C.elt({})

c=c+C.elt_from_cd({('X','X'):symbols('d')})

for i in range(1, 7):
    key = ('X', f'e_{i}','X')
    val = symbols(f'a_{i}')
    c = c + C.elt_from_cd({key: val})

    for j in range(1,7):
        key = ('X',f'e_{i}',f'e_{j}')
        val = symbols(f'b_{i}{j}')
        c = c + C.elt_from_cd({key: val})

    key = ('X', f'e_{i}','N')
    val = symbols(f'd_{i}')
    c = c + C.elt_from_cd({key: val})

    for j in range(i+1,7):
        key = (f'e_{i}',f'e_{j}','X')
        val = symbols(f'f_{i}{j}')
        c = c + C.elt_from_cd({key: val})

        for k in range(1,7):
            key = (f'e_{i}',f'e_{j}',f'e_{k}')
            val = symbols(f'g_{i}{j}{k}')
            c = c + C.elt_from_cd({key: val})

        key = (f'e_{i}',f'e_{j}','N')
        val = symbols(f'h_{i}{j}')
        c = c + C.elt_from_cd({key: val})

eqs=[]
ccb=c.cb()
d=3
for w in ccb.vd[d]:
    for i in range(len(ccb.vd[d][w])):
        if ccb.vd[d][w][i]!=0:
            eqs.append(ccb.vd[d][w][i])

solve(eqs)


{a_1: -2*g_166/5 - g_256 - 2*g_346/5,
 a_2: -2*g_266/5 - g_356,
 a_3: -9*g_366/5 - g_456,
 a_4: g_455,
 a_5: g_566,
 a_6: 0,
 b_11: -b_66 + h_26,
 b_12: b_56 - h_16 - h_25,
 b_13: -b_46 + h_15 + h_24,
 b_14: b_36 - h_14 - h_23,
 b_15: -b_26 + h_13,
 b_21: b_65 + h_36,
 b_22: -b_55 - h_26 - h_35,
 b_23: b_45 + h_25 + h_34,
 b_24: -b_35 - h_24,
 b_31: -b_64 + h_46,
 b_32: b_54 - h_36 - h_45,
 b_33: -b_44 + h_35,
 b_41: b_63 + h_56,
 b_42: -b_53 - h_46,
 b_51: -b_62,
 f_12: 0,
 f_13: 0,
 f_14: 0,
 f_15: 0,
 f_16: 0,
 f_23: 0,
 f_24: 0,
 f_25: 0,
 f_26: 0,
 f_34: 0,
 f_35: 0,
 f_36: 0,
 f_45: 0,
 f_46: 0,
 f_56: 0,
 g_121: 2*g_266,
 g_122: -3*g_166/5 - g_256 + 2*g_346/5,
 g_123: g_156 + 2*g_246,
 g_124: -g_236/2,
 g_125: g_136,
 g_131: 0,
 g_132: 8*g_266/5 - g_356,
 g_133: -g_166/5 + 4*g_346/5,
 g_134: g_156 + 2*g_246,
 g_135: -g_236/2,
 g_141: 0,
 g_142: -6*g_366/5 - g_456,
 g_143: 9*g_266/5,
 g_144: g_166/5 + g_346/5,
 g_145: g_156 + g_246,
 g_146: -3*g_236/2,
 g_151: 0,
 g_152: g_455,
 

In [106]:
c.subs(solve(eqs))

(b_56 - h_16 - h_25)*(X,e_1,e_2)+(b_45 + h_25 + h_34)*(X,e_2,e_3)+b_34*(X,e_3,e_4)+b_45*(X,e_4,e_5)+b_56*(X,e_5,e_6)+d_6*(X,e_6,N)+(g_156 + 2*g_246)*(e_1,e_2,e_3)+(g_156 + 2*g_246)*(e_1,e_3,e_4)+(g_156 + g_246)*(e_1,e_4,e_5)+g_156*(e_1,e_5,e_6)+h_16*(e_1,e_6,N)+g_246*(e_2,e_3,e_5)+g_246*(e_2,e_4,e_6)+h_25*(e_2,e_5,N)+h_34*(e_3,e_4,N)+(-2*g_166/5 - g_256 - 2*g_346/5)*(X,e_1,X)+(-b_66 + h_26)*(X,e_1,e_1)+(-b_55 - h_26 - h_35)*(X,e_2,e_2)+(-b_44 + h_35)*(X,e_3,e_3)+b_44*(X,e_4,e_4)+b_55*(X,e_5,e_5)+b_66*(X,e_6,e_6)+(-3*g_166/5 - g_256 + 2*g_346/5)*(e_1,e_2,e_2)+(-g_166/5 + 4*g_346/5)*(e_1,e_3,e_3)+(g_166/5 + g_346/5)*(e_1,e_4,e_4)+(3*g_166/5 - 2*g_346/5)*(e_1,e_5,e_5)+g_166*(e_1,e_6,e_6)+(g_256 + g_346)*(e_2,e_3,e_4)+(g_256 + g_346)*(e_2,e_4,e_5)+g_256*(e_2,e_5,e_6)+h_26*(e_2,e_6,N)+g_346*(e_3,e_4,e_6)+h_35*(e_3,e_5,N)+(-2*g_266/5 - g_356)*(X,e_2,X)+(b_65 + h_36)*(X,e_2,e_1)+(b_54 - h_36 - h_45)*(X,e_3,e_2)+b_43*(X,e_4,e_3)+b_54*(X,e_5,e_4)+b_65*(X,e_6,e_5)+2*g_266*(e_1,e_2,e_1)+(8*g_266/

In [22]:
e1.cast_as_ext_elt().wedge(X.cast_as_cochain())

(e_1,X)

In [48]:
for b in [e2]:#g.basis[3:len(g.basis)]:
    c1=C.elt({})
    for x in g.basis[3:len(g.basis)]:
        if b!=x: 
            temp=gE.elt_from_cd({(str(b),str(x)):1})
            c1=c1+temp.wedge((E.ad(x)).cast_as_cochain())
    if C.subspace_proj(c1,'harmonic')!=C.elt({}): print(b)

In [40]:
c1

-(e_1,e_2,e_1)+(e_2,e_3,e_3)+(e_2,e_4,e_4)+(e_2,e_5,e_5)+(e_2,e_6,e_6)+2*(e_2,N,N)

In [47]:
C.subspace_proj(c1,'harmonic')

0

# n=6 Case

In [4]:
g=Symp_symb(7)
C=cochain_complex(g)
gE=ext_alg(g)
Y,H,E,X,e1,e2,e3,e4,e5,e6,N=g.basis

In [89]:
T3=C.elt_from_cd({('e_1','e_4','e_2'):-3,('e_1','e_5','e_3'):-6,('e_1','e_6','e_4'):-5,
                  ('e_2','e_3','e_2'):3,('e_2','e_4','e_3'):3,('e_2','e_5','e_4'):-1,
                  ('e_2','e_6','e_5'):-5,('e_3','e_4','e_4'):4,('e_3','e_5','e_5'):4,
                  ('e_3','e_6','e_6'):-5,('e_4','e_5','e_6'):9})
T4=C.elt_from_cd({('X','e_4','e_1'):5,('X','e_5','e_2'):8,('X','e_6','e_3'):5})
T6=C.elt_from_cd({('X','e_6','e_1'):1})

In [90]:
# T3 is X-invariant
for a in [T3,T4,T6]:
    print('\nad(X)(',a,') :\n',X.ad(a))


ad(X)( -3*(e_1,e_4,e_2)-6*(e_1,e_5,e_3)-5*(e_1,e_6,e_4)+3*(e_2,e_3,e_2)+3*(e_2,e_4,e_3)-(e_2,e_5,e_4)-5*(e_2,e_6,e_5)+4*(e_3,e_4,e_4)+4*(e_3,e_5,e_5)-5*(e_3,e_6,e_6)+9*(e_4,e_5,e_6) ) :
 0

ad(X)( 5*(X,e_4,e_1)+8*(X,e_5,e_2)+5*(X,e_6,e_3) ) :
 -5*(X,e_3,e_1)-3*(X,e_4,e_2)+3*(X,e_5,e_3)+5*(X,e_6,e_4)

ad(X)( (X,e_6,e_1) ) :
 -(X,e_5,e_1)+(X,e_6,e_2)


In [ ]:
# T4, T6 are Y-invariant
for a in [T3,T4,T6]:
    print('\nad(Y)(',a,') :\n',Y.ad(a))
print()

# T3 is X-invariant
for a in [T3,T4,T6]:
    print('\nad(X))(',a,') :\n',X.ad(a))
print()

# However, the harmonic part is invariant in the sense that
# Harm_proj(Y.ad(K)) = 0...I think this is just coincidence, though
for a in [T3,T4,T6]:
    print(C.subspace_proj(Y.ad(a),'harmonic'))


ad(Y)( -3*(e_1,e_4,e_2)-6*(e_1,e_5,e_3)-5*(e_1,e_6,e_4)+3*(e_2,e_3,e_2)+3*(e_2,e_4,e_3)-(e_2,e_5,e_4)-5*(e_2,e_6,e_5)+4*(e_3,e_4,e_4)+4*(e_3,e_5,e_5)-5*(e_3,e_6,e_6)+9*(e_4,e_5,e_6) ) :
 -15*(e_1,e_4,e_1)-24*(e_1,e_5,e_2)-15*(e_1,e_6,e_3)+15*(e_2,e_3,e_1)+12*(e_2,e_4,e_2)-3*(e_2,e_5,e_3)-10*(e_2,e_6,e_4)+12*(e_3,e_4,e_3)+8*(e_3,e_5,e_4)-5*(e_3,e_6,e_5)+9*(e_4,e_5,e_5)

ad(Y)( 5*(X,e_4,e_1)+8*(X,e_5,e_2)+5*(X,e_6,e_3) ) :
 0

ad(Y)( (X,e_6,e_1) ) :
 0


ad(X))( -3*(e_1,e_4,e_2)-6*(e_1,e_5,e_3)-5*(e_1,e_6,e_4)+3*(e_2,e_3,e_2)+3*(e_2,e_4,e_3)-(e_2,e_5,e_4)-5*(e_2,e_6,e_5)+4*(e_3,e_4,e_4)+4*(e_3,e_5,e_5)-5*(e_3,e_6,e_6)+9*(e_4,e_5,e_6) ) :
 0

ad(X))( 5*(X,e_4,e_1)+8*(X,e_5,e_2)+5*(X,e_6,e_3) ) :
 -5*(X,e_3,e_1)-3*(X,e_4,e_2)+3*(X,e_5,e_3)+5*(X,e_6,e_4)

ad(X))( (X,e_6,e_1) ) :
 -(X,e_5,e_1)+(X,e_6,e_2)

0
0
0


#### Why does Wilc pass to a section of a line bundle?

The cells below suggest that the relevant terms must be related by Jacobi or something. \
Why should ad(Y)(K) have no harmonic component?

In [115]:
for c in C.basis(2,5):
    temp=C.subspace_proj(Y.ad(c),'harmonic')
    if temp!=C.elt({}): print('ad(Y)',c,'=',temp)

ad(Y) (X,e_5,e_1) = -5*(X,e_6,e_1)
ad(Y) (X,e_6,e_2) = 5*(X,e_6,e_1)


In [149]:
d=2
for w in range(1,10):
    M=C.subspace_basis('coclosed',d,w)
    for i in range(shape(M)[1]):
        temp=C.elt({d:{w:M.col(i)}})
        r=Y.ad(temp,mod='CE')
        if r!=C.subspace_proj(r,'coexact'): print(r)

In [150]:
d=2
for w in range(1,10):
    M=C.subspace_basis('closed',d,w)
    for i in range(shape(M)[1]):
        temp=C.elt({d:{w:M.col(i)}})
        r=X.ad(temp,mod='CE')
        if r!=C.subspace_proj(r,'exact'): print(r)

In [158]:
d=2
w=3
M=C.subspace_basis('closed',d,w)
i=0
temp=C.elt({d:{w:M.col(i)}})
r=X.ad(temp,mod='CE')
print(temp)
print(r)

-(X,e_4,e_2)+(X,e_5,e_3)
(X,e_3,e_2)-2*(X,e_4,e_3)+(X,e_5,e_4)


In [157]:
C.elt_from_cd({('e_4','e_2'):-1,('e_5','e_3'):1}).cb()

(X,e_3,e_2)-2*(X,e_4,e_3)+(X,e_5,e_4)

In [ ]:
K=C.regular_normal_2_cochain('K')
YK=Y.ad(K,mod='CE')
C.subspace_proj(YK,'harmonic')

In [120]:
c1=C.basis(2,5)[4]
c2=C.basis(2,5)[5]
C.dwi(('X','e_6','e_2'))

(2, 5, 5)

In [133]:
K.vd[2][5][4]*c1+K.vd[2][5][5]*c2

K[3, 9, 5]*(X,e_5,e_1)+K[3, 9, 5]*(X,e_6,e_2)

In [134]:
C.subspace_proj(c1,"coclosed")

1/2*(X,e_5,e_1)+1/2*(X,e_6,e_2)

In [135]:
C.subspace_proj(c2,"coclosed")

1/2*(X,e_5,e_1)+1/2*(X,e_6,e_2)

In [92]:
for w in C.basis(2):
    for a in range(shape(C.subspace_basis('closed',1,w))[1]):
        c=C.elt({1:{w:(C.subspace_basis('closed',1,w).col(a))}})
        if C.cb(X.ad(c))!=C.elt({}): print('\n',c)

In [ ]:
## Is harm_proj(im(ad(Y))) nonzero? Which lines are contained in it? <T4,T6>, the Wilcz subspace
im_Y_list=[]
for w in C.basis(2):
    for c in C.basis(2,w):
        elt=C.subspace_proj(Y.ad(c),'harmonic')
        if elt!=C.elt({}):
            im_Y_list.append(elt)

In [ ]:
im_Y_list

[-5/2*(X,e_4,e_1)-4*(X,e_5,e_2)-5/2*(X,e_6,e_3),
 -5/6*(X,e_4,e_1)-4/3*(X,e_5,e_2)-5/6*(X,e_6,e_3),
 5/6*(X,e_4,e_1)+4/3*(X,e_5,e_2)+5/6*(X,e_6,e_3),
 5/2*(X,e_4,e_1)+4*(X,e_5,e_2)+5/2*(X,e_6,e_3),
 -5*(X,e_6,e_1),
 5*(X,e_6,e_1)]

In [99]:

im_X_list=[]
for w in C.basis(2):
    if w>1:
        for c in C.basis(2,w):
            elt=C.subspace_proj(X.ad(c),'harmonic')
            if elt!=C.elt({}):
                im_X_list.append(elt)
im_X_list

[1/42*(e_1,e_4,e_2)+1/21*(e_1,e_5,e_3)+5/126*(e_1,e_6,e_4)-1/42*(e_2,e_3,e_2)-1/42*(e_2,e_4,e_3)+1/126*(e_2,e_5,e_4)+5/126*(e_2,e_6,e_5)-2/63*(e_3,e_4,e_4)-2/63*(e_3,e_5,e_5)+5/126*(e_3,e_6,e_6)-1/14*(e_4,e_5,e_6),
 1/42*(e_1,e_4,e_2)+1/21*(e_1,e_5,e_3)+5/126*(e_1,e_6,e_4)-1/42*(e_2,e_3,e_2)-1/42*(e_2,e_4,e_3)+1/126*(e_2,e_5,e_4)+5/126*(e_2,e_6,e_5)-2/63*(e_3,e_4,e_4)-2/63*(e_3,e_5,e_5)+5/126*(e_3,e_6,e_6)-1/14*(e_4,e_5,e_6),
 1/42*(e_1,e_4,e_2)+1/21*(e_1,e_5,e_3)+5/126*(e_1,e_6,e_4)-1/42*(e_2,e_3,e_2)-1/42*(e_2,e_4,e_3)+1/126*(e_2,e_5,e_4)+5/126*(e_2,e_6,e_5)-2/63*(e_3,e_4,e_4)-2/63*(e_3,e_5,e_5)+5/126*(e_3,e_6,e_6)-1/14*(e_4,e_5,e_6),
 -3/70*(e_1,e_4,e_2)-3/35*(e_1,e_5,e_3)-1/14*(e_1,e_6,e_4)+3/70*(e_2,e_3,e_2)+3/70*(e_2,e_4,e_3)-1/70*(e_2,e_5,e_4)-1/14*(e_2,e_6,e_5)+2/35*(e_3,e_4,e_4)+2/35*(e_3,e_5,e_5)-1/14*(e_3,e_6,e_6)+9/70*(e_4,e_5,e_6),
 -2/105*(e_1,e_4,e_2)-4/105*(e_1,e_5,e_3)-2/63*(e_1,e_6,e_4)+2/105*(e_2,e_3,e_2)+2/105*(e_2,e_4,e_3)-2/315*(e_2,e_5,e_4)-2/63*(e_2,e_6,e_5)+8/3

Note that the closed forms (and cohomology) form a $G_-$-module, but not a $G_+$

## ad(X) kernel

In [8]:
X_ker1=compute_ad_kernel(X,1)
X_ker2=compute_ad_kernel(X,2)

In [9]:
for w in X_ker1:
    print('\n',w)
    for t in X_ker1[w]:
        print(t,'-->',(compute_eigenval(H,t),compute_eigenval(E,t)))


 2
-1/2*(e_1,Y)-1/2*(e_2,H)+(e_3,X) --> (3, -1)

 1
(X,E) --> (-2, 0)
(e_1,E) --> (5, -1)
-1/2*(e_1,H)+(e_2,X) --> (5, -1)
(N,e_6) --> (5, -1)

 0
(X,X) --> (0, 0)
(e_1,X) --> (7, -1)
(e_1,e_1)+(e_2,e_2)+(e_3,e_3)+(e_4,e_4)+(e_5,e_5)+(e_6,e_6) --> (0, 0)
(N,N) --> (0, 0)

 -1
(e_1,e_2)+(e_2,e_3)+(e_3,e_4)+(e_4,e_5)+(e_5,e_6) --> (2, 0)

 -2
(e_1,e_3)+(e_2,e_4)+(e_3,e_5)+(e_4,e_6) --> (4, 0)

 -3
(e_1,e_4)+(e_2,e_5)+(e_3,e_6) --> (6, 0)

 -4
(e_1,e_5)+(e_2,e_6) --> (8, 0)

 -5
(X,e_6) --> (3, 1)
(e_1,e_6) --> (10, 0)

 -6
(X,N) --> (-2, 2)
(e_1,N) --> (5, 1)

 6
(N,X) --> (2, -2)

 7
(N,E) --> (0, -2)


In [10]:
for w in X_ker2:
    print('\n',w)
    for t in X_ker2[w]:
        print(t,'-->',(compute_eigenval(H,t),compute_eigenval(E,t)))


 3
-1/2*(X,e_1,Y)-1/2*(X,e_2,H)+(X,e_3,X) --> (1, -1)
(e_1,e_2,E) --> (8, -2)
-1/2*(e_1,e_2,H)+(e_1,e_3,X) --> (8, -2)
(e_1,N,e_5)+(e_2,N,e_6) --> (8, -2)
-1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6) --> (1, -1)

 2
(X,e_1,E) --> (3, -1)
-1/2*(X,e_1,H)+(X,e_2,X) --> (3, -1)
(X,N,e_6) --> (3, -1)
(e_1,e_2,X) --> (10, -2)
(e_1,N,e_6) --> (10, -2)
4*(e_1,e_2,e_1)+4*(e_1,e_3,e_2)+3*(e_1,e_4,e_3)+2*(e_1,e_5,e_4)+(e_1,e_6,e_5)+(e_2,e_3,e_3)+(e_2,e_4,e_4)+(e_2,e_5,e_5)+(e_2,e_6,e_6) --> (3, -1)
5*(e_1,e_2,e_1)+5*(e_1,e_3,e_2)+3*(e_1,e_4,e_3)+(e_1,e_5,e_4)+2*(e_2,e_3,e_3)+2*(e_2,e_4,e_4)+(e_2,e_5,e_5)+(e_3,e_4,e_5)+(e_3,e_5,e_6) --> (3, -1)

 1
(X,e_1,X) --> (5, -1)
(X,e_1,e_1)+(X,e_2,e_2)+(X,e_3,e_3)+(X,e_4,e_4)+(X,e_5,e_5)+(X,e_6,e_6) --> (-2, 0)
(X,N,N) --> (-2, 0)
(e_1,e_2,e_2)+(e_1,e_3,e_3)+(e_1,e_4,e_4)+(e_1,e_5,e_5)+(e_1,e_6,e_6) --> (5, 

In [11]:
for w in X_ker1:
    print('\n',w)
    for t in X_ker1[w]:
        print(t,'-->',C.subspace_proj(t,'harmonic'))


 2
-1/2*(e_1,Y)-1/2*(e_2,H)+(e_3,X) --> 0

 1
(X,E) --> 0
(e_1,E) --> 0
-1/2*(e_1,H)+(e_2,X) --> 0
(N,e_6) --> 0

 0
(X,X) --> 0
(e_1,X) --> 0
(e_1,e_1)+(e_2,e_2)+(e_3,e_3)+(e_4,e_4)+(e_5,e_5)+(e_6,e_6) --> 0
(N,N) --> 0

 -1
(e_1,e_2)+(e_2,e_3)+(e_3,e_4)+(e_4,e_5)+(e_5,e_6) --> 0

 -2
(e_1,e_3)+(e_2,e_4)+(e_3,e_5)+(e_4,e_6) --> 0

 -3
(e_1,e_4)+(e_2,e_5)+(e_3,e_6) --> (e_1,e_4)+(e_2,e_5)+(e_3,e_6)

 -4
(e_1,e_5)+(e_2,e_6) --> 0

 -5
(X,e_6) --> 0
(e_1,e_6) --> (e_1,e_6)

 -6
(X,N) --> (X,N)
(e_1,N) --> 0

 6
(N,X) --> 0

 7
(N,E) --> 0


In [12]:
Ch1_basis=[]
j=1
Ch2_basis=[]
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch1_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

Ch2_basis=[]
j=2
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch2_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

In [13]:
for c in Ch2_basis:
    print('\n',c,'-->',(compute_eigenval(H,c),compute_eigenval(E,c)))


 -1/3*(e_1,e_4,e_2)-2/3*(e_1,e_5,e_3)-5/9*(e_1,e_6,e_4)+1/3*(e_2,e_3,e_2)+1/3*(e_2,e_4,e_3)-1/9*(e_2,e_5,e_4)-5/9*(e_2,e_6,e_5)+4/9*(e_3,e_4,e_4)+4/9*(e_3,e_5,e_5)-5/9*(e_3,e_6,e_6)+(e_4,e_5,e_6) --> (1, -1)

 3/5*(e_1,e_2,e_3)+3/5*(e_1,e_3,e_4)-2/5*(e_1,e_4,e_5)-7/5*(e_1,e_5,e_6)+(e_2,e_3,e_5)+(e_2,e_4,e_6) --> (7, -1)

 -1/2*(e_1,e_2,e_4)-1/2*(e_1,e_3,e_5)-3/2*(e_1,e_4,e_6)+(e_2,e_3,e_6) --> (9, -1)

 (e_1,e_2,e_5)+(e_1,e_3,e_6) --> (11, -1)

 (e_1,e_2,e_6) --> (13, -1)

 (X,e_4,e_1)+8/5*(X,e_5,e_2)+(X,e_6,e_3) --> (-8, 0)

 (X,e_6,e_1) --> (-12, 0)


In [14]:
Ch2_basis_wghts=[]
Ch2_plus_basis=[]
for i in range(len(Ch2_basis)):
    c=Ch2_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch2_basis_wghts.append(w)
    if w>0: Ch2_plus_basis.append(c)

Ch1_basis_wghts=[]
Ch1_plus_basis=[]
for i in range(len(Ch1_basis)):
    c=Ch1_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch1_basis_wghts.append(w)
    if w>0: Ch1_plus_basis.append(c)

In [15]:
print(Ch2_basis_wghts)
print(len(Ch2_basis))

[3, 0, -1, -2, -3, 4, 6]
7


In [20]:
for k in range(len(Ch2_plus_basis)):
    c=Ch2_plus_basis[k]
    i=0
    while c!=C.elt({}):
        c=X.ad(c)
        i+=1
    j=0
    c=Ch2_plus_basis[k]
    while c!=C.elt({}):
        c=Y.ad(c)
        j+=1
    c=Ch2_plus_basis[k]
    print(k,':',(compute_eigenval(H,c),compute_eigenval(E,c)),',  X^{}(c) = 0, Y^{}(c) = 0'.format(i,j),)

0 : (1, -1) ,  X^1(c) = 0, Y^2(c) = 0
1 : (-8, 0) ,  X^7(c) = 0, Y^1(c) = 0
2 : (-12, 0) ,  X^11(c) = 0, Y^1(c) = 0


## Weight Plots

In [ ]:
%matplotlib notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
C2_wghts=set()
for w in C.basis(2):
    for c in C.basis(2,w):
        C2_wghts.add((compute_eigenval(H,c),compute_eigenval(E,c)))

In [ ]:
C2_wghts

In [ ]:
x_pts=np.array([a[0] for a in C2_wghts])
y_pts=np.array([a[1] for a in C2_wghts])

In [ ]:
plt.plot(x_pts,y_pts,'o')
plt.show()

In [ ]:
plt.plot(0,0,'bo')
plt.grid(visible=True)
for wght in C2_wghts:
    plt.plot(wght[0],wght[1],'ro')

In [ ]:
for w in C.basis(2):
    print(w,'-->',shape(C.subspace_basis('harmonic',2,w))[1])

In [ ]:
for i in range(C.subspace_basis('harmonic',2,3).shape[1]):
    print(C.elt({2:{3:C.subspace_basis('harmonic',2,3).col(i)}}),'\n')

In [ ]:
for i in range(C.subspace_basis('harmonic',2,4).shape[1]):
    print(C.elt({2:{4:C.subspace_basis('harmonic',2,4).col(i)}}),'\n')

In [ ]:
for i in range(C.subspace_basis('harmonic',2,6).shape[1]):
    print(C.elt({2:{6:C.subspace_basis('harmonic',2,6).col(i)}}),'\n')

In [ ]:
f=C.basis(3,0)[6]
n=2
h=C.basis(2,4)[22]
m=2

w=gE.basis(4,10)[3]
print(w)
Gerst_prod(h,f,w)

In [ ]:
from itertools import permutations 
for a in permutations([0,1,2]):
    print(a)

### How does ad affect weights?

In [ ]:
Y_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        Yc=Y.ad(c)
        temp2=(compute_eigenval(H,Yc),compute_eigenval(E,Yc))
        if Yc!=C.elt({}):
            if temp1 in Y_op_dict: 
                if temp2 !=Y_op_dict[temp1]: print(temp1)
            else: Y_op_dict[temp1]=temp2
for t in Y_op_dict:
    if Y_op_dict[t]!=tuple([t[0]-2,t[1]]):print(t)

In [ ]:
X_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        Xc=X.ad(c)
        temp2=(compute_eigenval(H,Xc),compute_eigenval(E,Xc))
        if Xc!=C.elt({}):
            if temp1 in X_op_dict: 
                if temp2 !=X_op_dict[temp1]: print(temp1)
            else: X_op_dict[temp1]=temp2

for t in X_op_dict:
    if X_op_dict[t]!=tuple([t[0]+2,t[1]]):print(t)

In [ ]:
e1_op_dict={}
for w in C.basis(2):
    for c in C.basis(2,w):
        temp1=(compute_eigenval(H,c),compute_eigenval(E,c))
        e1c=e1.ad(c)
        temp2=(compute_eigenval(H,e1c),compute_eigenval(E,e1c))
        if e1c!=C.elt({}):
            if temp1 in e1_op_dict: 
                if temp2 !=e1_op_dict[temp1]: print(temp1)
            else: e1_op_dict[temp1]=temp2

for t in e1_op_dict:
    if e1_op_dict[t]!=tuple([t[0]-5,t[1]+1]):print(t)

In [ ]:
for c in Ch2_basis:
    print((compute_eigenval(H,c),compute_eigenval(E,c)))

In [ ]:
for w in C.basis(2):
    print('\n\n',w)
    wght_set=set()
    for c in C.basis(2,w):
        wght_set.add((compute_eigenval(H,c),compute_eigenval(E,c)))
    print(wght_set)

In [ ]:
for w in C.basis(2):
    for i in range(shape(C.subspace_basis('harmonic',2,w))[1]):
        temp=C.elt({2:{w:C.subspace_basis('harmonic',2,w).col(i)}})
        print(temp)
        # if C.subspace_proj(X.ad(temp),'harmonic')!=C.elt({}):
        #     print(temp)

In [ ]:
for w in X_ker2:
    print('\n',w)
    for t in X_ker2[w]:
        th=C.subspace_proj(t,'harmonic')
        if th!=C.elt({}) and th-t!=C.elt({}):
            print(t)
        # print(t,'-->',C.subspace_proj(t,'harmonic'))

In [ ]:
for w in [0]:
    print('\n',w)
    for t in X_ker2[w]:
        # th=C.subspace_proj(t,'harmonic')
        # if th!=C.elt({}) and th-t!=C.elt({}):
        #     print(t)
        print(t,'-->',C.subspace_proj(t,'harmonic'))


In [ ]:
X.ad(C.subspace_proj(X_ker2[0][2],'harmonic'))

In [ ]:
C.subspace_proj(X_ker2[0][2],'harmonic')==-Rational(7,5)*X_ker2[0][1]+X_ker2[0][2]

In [ ]:
nW=C.elt_from_cd({('e_1','e_4','e_2'):3,('e_1','e_5','e_3'):6,('e_1','e_6','e_4'):5,('e_2','e_3','e_2'):-3,
                  ('e_2','e_4','e_3'):-3,('e_2','e_5','e_4'):1,('e_2','e_6','e_5'):5,('e_3','e_4','e_4'):-4,
                  ('e_3','e_5','e_5'):-4,('e_3','e_6','e_6'):5,('e_4','e_5','e_6'):-9})

In [ ]:
alpha=g.ext_alg.elt_from_cd({('e_1','e_3'):Rational(1,3)})
alpha

In [ ]:
for i in range(6):
    temp=alpha
    for j in range(i):
        temp=-X.ad(temp)
    print(temp,' *  e',6-i)

In [ ]:
Inv_V_GL2=[C.elt_from_cd({('e_1','E'):1}),C.elt_from_cd({('e_1','X'):1}),
           C.elt_from_cd({('e_2','X'):2,('e_1','H'):-1}),C.elt_from_cd({('e_3','X'):2,('e_2','H'):-1,('e_1','Y'):-1})]

In [ ]:
for A in Inv_V_GL2:
    print(C.subspace_proj(A,'harmonic'))
    print(C.subspace_proj(A.cb(),'harmonic'))

In [ ]:
print(X.ad(nW,mod='CE'))
nW

In [ ]:
alpha_cd={('e_3','e_6'):5,('e_4','e_5'):-9}
alpha=g.ext_alg.elt_from_cd({})
alpha+=g.ext_alg.elt_from_cd(alpha_cd)
X_alpha=C.elt({})
for i in range(6):
    temp=alpha
    for j in range(i):
        temp=-X.ad(temp,mod='E')
    X_alpha=X_alpha+temp.tensor(g.basis[9-i])

In [ ]:
compute_eigenval(E,alpha)

In [ ]:
X_alpha

In [ ]:
nW-X_alpha

In [ ]:
for i in range(6):
    ei=[e1,e2,e3,e4,e5,e6][6-i-1]
    w=3-ei.wght+i
    print('\n',ei)
    for A in g.ext_alg.basis(2,w):
        if set([str(a) for a in A.components]).issubset({'e_1','e_2','e_3','e_4','e_5','e_6'}):
            temp=A
            for j in range(i):
                temp=-X.ad(temp,mod='E')
            print(A,'-->',temp)

# n=7 Case

In [34]:
g=Symp_symb(9)
C=cochain_complex(g)
gE=ext_alg(g)
Y,H,E,X,e1,e2,e3,e4,e5,e6,e7,e8,N=g.basis

In [35]:
Ch1_basis=[]
j=1
Ch2_basis=[]
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch1_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

Ch2_basis=[]
j=2
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch2_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

In [36]:
Ch2_basis_wghts=[]
Ch2_plus_basis=[]
for i in range(len(Ch2_basis)):
    c=Ch2_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch2_basis_wghts.append(w)
    if w>0: Ch2_plus_basis.append(c)

Ch1_basis_wghts=[]
Ch1_plus_basis=[]
for i in range(len(Ch1_basis)):
    c=Ch1_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch1_basis_wghts.append(w)
    if w>0: Ch1_plus_basis.append(c)

In [37]:
print(Ch2_basis_wghts)
print(len(Ch2_basis))

[3, 2, 1, 0, -1, -1, -2, -2, -3, -4, -5, 4, 4, 6, 8]
15


In [38]:
for k in range(len(Ch2_plus_basis)):
    c=Ch2_plus_basis[k]
    i=0
    while c!=C.elt({}):
        c=X.ad(c)
        i+=1
    j=0
    c=Ch2_plus_basis[k]
    while c!=C.elt({}):
        c=Y.ad(c)
        j+=1
    c=Ch2_plus_basis[k]
    print(k,':',(compute_eigenval(H,c),compute_eigenval(E,c)),',  X^{}(c) = 0, Y^{}(c) = 0'.format(i,j),)

0 : (3, -1) ,  X^1(c) = 0, Y^4(c) = 0
1 : (5, -1) ,  X^1(c) = 0, Y^6(c) = 0
2 : (7, -1) ,  X^1(c) = 0, Y^8(c) = 0
3 : (-8, 0) ,  X^7(c) = 0, Y^1(c) = 0
4 : (1, -1) ,  X^1(c) = 0, Y^2(c) = 0
5 : (-12, 0) ,  X^11(c) = 0, Y^1(c) = 0
6 : (-16, 0) ,  X^15(c) = 0, Y^1(c) = 0


In [39]:
C2_wghts=set()
for w in C.basis(2):
    for c in C.basis(2,w):
        C2_wghts.add((compute_eigenval(H,c),compute_eigenval(E,c)))

# n=9 Case (takes forever)

In [32]:
g=Symp_symb(11)
C=cochain_complex(g)
gE=ext_alg(g)
Y,H,E,X,e1,e2,e3,e4,e5,e6,e7,e8,e9,e10,N=g.basis

In [33]:
Ch1_basis=[]
j=1
Ch2_basis=[]
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch1_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

Ch2_basis=[]
j=2
for w in C.basis(j):
    for i in range(shape(C.subspace_basis('harmonic',j,w))[1]):
        Ch2_basis.append(C.elt({j:{w:C.subspace_basis('harmonic',j,w).col(i)}}))

KeyboardInterrupt: 

In [ ]:
Ch2_basis_wghts=[]
Ch2_plus_basis=[]
for i in range(len(Ch2_basis)):
    c=Ch2_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch2_basis_wghts.append(w)
    if w>0: Ch2_plus_basis.append(c)

Ch1_basis_wghts=[]
Ch1_plus_basis=[]
for i in range(len(Ch1_basis)):
    c=Ch1_basis[i]
    w=None
    for j in range(-10,10):
        if c.wght_proj(j)==c: w=j
    if w==None: print('element', i,'weight not computed')
    Ch1_basis_wghts.append(w)
    if w>0: Ch1_plus_basis.append(c)

In [ ]:
print(Ch2_basis_wghts)
print(len(Ch2_basis))

[3, 2, 1, 0, -1, -1, -2, -2, -3, -4, -5, 4, 4, 6, 8]
15


In [ ]:
for k in range(len(Ch2_plus_basis)):
    c=Ch2_plus_basis[k]
    i=0
    while c!=C.elt({}):
        c=X.ad(c)
        i+=1
    j=0
    c=Ch2_plus_basis[k]
    while c!=C.elt({}):
        c=Y.ad(c)
        j+=1
    c=Ch2_plus_basis[k]
    print(k,':',(compute_eigenval(H,c),compute_eigenval(E,c)),',  X^{}(c) = 0, Y^{}(c) = 0'.format(i,j),)

0 : (3, -1) ,  X^1(c) = 0, Y^4(c) = 0
1 : (5, -1) ,  X^1(c) = 0, Y^6(c) = 0
2 : (7, -1) ,  X^1(c) = 0, Y^8(c) = 0
3 : (-8, 0) ,  X^7(c) = 0, Y^1(c) = 0
4 : (1, -1) ,  X^1(c) = 0, Y^2(c) = 0
5 : (-12, 0) ,  X^11(c) = 0, Y^1(c) = 0
6 : (-16, 0) ,  X^15(c) = 0, Y^1(c) = 0


In [ ]:
C2_wghts=set()
for w in C.basis(2):
    for c in C.basis(2,w):
        C2_wghts.add((compute_eigenval(H,c),compute_eigenval(E,c)))

In [ ]:
C2_wghts